After constructiong the final phase 01 dataset, adding circuit features, i will try to predict the Japan 2026 GP Results, to test the model/dataset

In [5]:
import fastf1
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from xgboost import XGBClassifier, XGBRegressor, plot_importance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
import shap
import matplotlib.pyplot as plt

In [6]:
data = pd.read_parquet('/Users/dimkostir/Desktop/Projects/f1-predictions/data/processed/final_phase_01/final_01_circuits.parquet')

In [7]:
data['Year'].value_counts().sort_index()

Year
2022    440
2023    440
2024    479
2025    479
2026     66
Name: count, dtype: int64

In [8]:
print(f"FP2 features shape: {data.shape}")
print(f"Master Phase 2 shape: {data.shape}")
print(f"\nColumns in master2:\n{data.columns.tolist()}")
print(f"\nNull counts:\n{data.isnull().sum()}")

FP2 features shape: (1904, 19)
Master Phase 2 shape: (1904, 19)

Columns in master2:
['Driver', 'TeamName_x', 'GridPosition', 'Finish_Position', 'Status', 'delta_to_pole', 'Qual_Position', 'Year', 'Location', 'fp2_median_pace', 'fp2_team_dif', 'fp2_deg_rate', 'circuit_length', 'num_corners', 'num_drs_zones', 'avg_speed_kmh', 'safety_car_rate', 'track_type', 'overtaking_ability']

Null counts:
Driver                  0
TeamName_x              0
GridPosition            2
Finish_Position         2
Status                  0
delta_to_pole          22
Qual_Position           4
Year                    0
Location                0
fp2_median_pace       476
fp2_team_dif          501
fp2_deg_rate          583
circuit_length        440
num_corners           440
num_drs_zones         440
avg_speed_kmh         440
safety_car_rate       440
track_type            440
overtaking_ability    440
dtype: int64


In [9]:
training_data = data.query("Year == 2024 or Year == 2025 or Year == 2022 or Year == 2023 or (Year == 2026 and Location != 'Suzuka')")
training_data.shape

(1882, 19)

In [10]:
training_data['Year'].value_counts().sort_index()

Year
2022    440
2023    440
2024    479
2025    479
2026     44
Name: count, dtype: int64

In [11]:
test_data = data.query("Year == 2026 and Location == 'Suzuka'")
print(test_data.shape)
test_data.head(10)

(22, 19)


,Driver,TeamName_x,GridPosition,Finish_Position,Status,delta_to_pole,Qual_Position,Year,Location,fp2_median_pace,fp2_team_dif,fp2_deg_rate,circuit_length,num_corners,num_drs_zones,avg_speed_kmh,safety_car_rate,track_type,overtaking_ability
1882,ANT,Mercedes,1.0,1.0,Finished,0.000,1.0,2026,Suzuka,94.5380,-0.3560,-1.453595,5807.0,18.0,2.0,230.0,0.12,permanent,0.4
1883,PIA,McLaren,3.0,2.0,Finished,0.354,3.0,2026,Suzuka,95.4640,-11.7900,2.886446,5807.0,18.0,2.0,230.0,0.12,permanent,0.4
1884,LEC,Ferrari,4.0,3.0,Finished,0.627,4.0,2026,Suzuka,95.2810,-0.8065,-0.305581,5807.0,18.0,2.0,230.0,0.12,permanent,0.4
1885,RUS,Mercedes,2.0,4.0,Finished,0.298,2.0,2026,Suzuka,94.8940,0.3560,0.094857,5807.0,18.0,2.0,230.0,0.12,permanent,0.4
1886,NOR,McLaren,5.0,5.0,Finished,0.631,5.0,2026,Suzuka,107.2540,11.7900,4.116194,5807.0,18.0,2.0,230.0,0.12,permanent,0.4
1887,HAM,Ferrari,6.0,6.0,Finished,0.789,6.0,2026,Suzuka,96.0875,0.8065,4.730494,5807.0,18.0,2.0,230.0,0.12,permanent,0.4
1888,GAS,Alpine,7.0,7.0,Finished,0.913,7.0,2026,Suzuka,96.1780,-0.4825,0.484922,5807.0,18.0,2.0,230.0,0.12,permanent,0.4
1889,VER,Red Bull Racing,11.0,8.0,Finished,1.484,11.0,2026,Suzuka,96.1220,-0.0300,0.437018,5807.0,18.0,2.0,230.0,0.12,permanent,0.4
1890,LAW,Racing Bulls,14.0,9.0,Finished,1.717,14.0,2026,Suzuka,96.9640,NaN,0.134022,5807.0,18.0,2.0,230.0,0.12,permanent,0.4
1891,OCO,Haas F1 Team,12.0,10.0,Finished,1.531,12.0,2026,Suzuka,96.0000,-0.1840,0.302553,5807.0,18.0,2.0,230.0,0.12,permanent,0.4


In [12]:
print(training_data['Finish_Position'].isnull().sum())
print(test_data['Finish_Position'].isnull().sum())

2
0


In [15]:
le = LabelEncoder()
# Fit on training
training_data["track_type"] = le.fit_transform(training_data["track_type"])

# Transform test
test_data["track_type"] = le.transform(test_data["track_type"])

training_clean = training_data[training_data["Finish_Position"].notna()].copy()

/var/folders/s0/k6p2569n6w1br5pz86sgwm400000gn/T/ipykernel_5945/2063686892.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  training_data["track_type"] = le.fit_transform(training_data["track_type"])
/var/folders/s0/k6p2569n6w1br5pz86sgwm400000gn/T/ipykernel_5945/2063686892.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data["track_type"] = le.transform(test_data["track_type"])


In [21]:
X_train = training_clean.drop(columns = ["Driver", "TeamName_x", "Status", "Year", "Location", "Finish_Position"], axis =1)
y_train = training_clean["Finish_Position"].dropna().astype(int)

X_test = test_data.drop(columns = ["Driver", "TeamName_x", "Status", "Year", "Location", "Finish_Position"], axis = 1)
y_test = test_data["Finish_Position"].astype(int)

bst = XGBRegressor(n_estimators = 200, max_depth = 4, learning_rate = 0.01)
bst.fit(X_train, y_train)

japan_2026_predictions = bst.predict(X_test)

In [22]:
print("Training Complete!\n")
print(f"Japan 2026 Predictions:{japan_2026_predictions}")

Training Complete!

Japan 2026 Predictions:[ 3.613703   5.307993   5.825689   5.6904593  6.9972157  7.4601555
  8.914211  11.685094  13.172145  11.988223  13.10836   10.227491
 10.264703  10.931685  13.818566  13.149496  13.686269  13.759709
 14.198654  13.392234  13.954228  14.837139 ]


In [24]:
japan_results = pd.DataFrame({
    'Driver': test_data['Driver'].values,
    'Grid Position': test_data['GridPosition'].values,
    'Actual_finish': test_data['Finish_Position'].values,
    'Predicted_score': japan_2026_predictions
})

japan_results['Predicted Finish'] = japan_results['Predicted_score'].rank(method = 'dense').astype(int)
japan_results.sort_values('Actual_finish')

japan_results

,Driver,Grid Position,Actual_finish,Predicted_score,Predicted Finish
0,ANT,1.0,1.0,3.613703,1
1,PIA,3.0,2.0,5.307993,2
2,LEC,4.0,3.0,5.825689,4
3,RUS,2.0,4.0,5.690459,3
4,NOR,5.0,5.0,6.997216,5
5,HAM,6.0,6.0,7.460155,6
6,GAS,7.0,7.0,8.914211,7
7,VER,11.0,8.0,11.685094,11
8,LAW,14.0,9.0,13.172145,15
9,OCO,12.0,10.0,11.988223,12
